# Currency, party relationships, and payment mechanisms

This notebook extends the currency/party EDA in notebooks 02--04. It does not repeat their basic routing charts. Implementation details are in `src/remaining_analysis.py`, particularly `matched_effect` and `currency_party`.

The input is the preserved cleaned CSV. A source-SHA-keyed private compact cache avoids repeated text scans. No transaction is dropped, deduplicated, imputed, or winsorized. Analytic contrasts restrict populations explicitly and report common support. B and C remain source codes: their economic meaning must be confirmed from organizer documentation before assigning customer/business semantics.

All 95% intervals assume independent hypothetical draws from the synthetic generator. Observed finite-population counts have no sampling error. Intervals do not establish real-world generalization. Multiple-comparison risk applies to all exploratory categories; association p-values have BH adjustment and effects are reported for every predeclared contrast.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "data/cleaned/swift_payments_cleaned.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / ".venv/Scripts/python.exe").exists()
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve(), "Use the project virtual environment"
sys.path.insert(0, str(ROOT / "src"))
from remaining_analysis import Study, currency_party
study = Study(ROOT, '05_currency_party', prerequisites=())

## Segment composition and effect sizes

**Purpose.** Establish whether currency and party composition are related.

**Hypothesis.** Currency distributions and routing composition vary by party pair.

**Methodology.** Full-population currency-by-party profiles, conditional route rates with Wilson intervals, and Cramér's V. Route rate denominator is currency × party population; V measures overall association strength, not direction.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** Use V to judge magnitude, and profiles to locate the conditions; small p-values alone are insufficient.

**Limitations.** Sparse currencies can have unstable rates. B/C semantics and bank/country composition remain unresolved.

**Business Implications.** Segment-aware baselines can prevent misleading comparisons driven by portfolio composition.

In [ ]:
tables, associations, duration_effects, covered_effects = currency_party(study)
display(associations)
display(tables["party_profile"])
display(tables["currency_party_profile"].sort_values("n",ascending=False).head(24))

## Adjusted duration and Covered routing

**Purpose.** Separate portfolio mix from within-segment differences.

**Hypothesis.** USD versus other currencies and party pairs versus C_C retain duration differences on common support; currency retains an association with Covered selection among non-direct payments.

**Methodology.** Predeclared USD and C_C references. Exact strata control route, participants, FX, fixed USD amount bands and either party or currency. Compare mean log(1+duration); min-arm overlap weights avoid extrapolation. Report all contrasts at minimum 20 and 100 per arm. Covered contrasts for USD/EUR/GBP/CHF control participants, FX, party, amount band among non-direct payments. The binary difference is a risk difference in rate units (×100 for percentage points). Independent-cell analytic 95% intervals treat weights as fixed.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** expm1(adjusted log difference) is a relative difference in geometric means of duration+1, not a median difference or percent change in raw seconds. Direction reversals across strata are counterexamples to universal rules.

**Limitations.** Strata exclude unsupported groups only from comparisons, not from source profiles. Amount bands leave residual variation; no bank or date adjustment here. Alternatives include bank composition and generator branches.

**Business Implications.** Use observed segment baselines to prioritize investigation; no causal routing benefit or savings is established.

In [ ]:
display(duration_effects)
display(covered_effects)
display(Image(filename=str(study.charts / "party_duration.png")))
display(Image(filename=str(study.charts / "currency_duration.png")))

In [ ]:
report = study.finish([f"Currency-party Cramér V = {associations.iloc[0].cramers_v:.6f}; population = {len(study.df):,}.",
                       *[f"{r.contrast} {r.level}, min arm {r.min_cell_per_arm}: adjusted log-duration difference {getattr(r, 'adjusted_difference', float('nan')):.6f}; shared strata {r.shared_strata}." for r in duration_effects.itertuples()],
                       "Universal currency/party duration rules require revision wherever within-stratum differences reverse; all directions and support counts are exported."], ['Associations are descriptive and sensitive to residual bank/time composition.', 'Sparse groups and no-overlap strata cannot support adjusted claims; small groups are flagged.', 'Currency denomination is not an observed FX rate; FX count alone cannot identify conversion costs.'], 'Test observed anomaly categories against an unlabeled comparison population, preserving CUSTOMER_RECALL.')
display(Markdown("### Final findings and limitations\n" + "\n".join("- " + x for x in report["findings"]) + "\n\n" + " ".join(report["limitations"])))
display(pd.DataFrame(study.checks))
display(Markdown("### Completion summary\nExecution passed for " + format(report["population"], ",") + " transactions. All generated files are listed below. No unresolved material errors.\n\n" + "\n".join("- `"+x+"`" for x in report["files"]) + "\n\nRecommended next investigation: " + report["next_investigation"]))